In [1]:
import pandas as pd
from neutrophil_shape.config.loader import load_config
from neutrophil_shape.CustomFunctions import DetailedBalance, utils

In [6]:
### load config stuff
config = load_config(microscope_type = 'confocal')
#set alignment
config._alignment = 'trajectory'
savedir = config.common.savedir
datadir = savedir / 'shape_data'
dbdir = savedir / 'detailed_balance'
npcs = config.common.npcs
ntrans = config.db_params.ntrans
origins = config.db_params.origins
pc_combos = config.common.pc_combos
####### load common directories and data
FullFrame = pd.read_csv(datadir.joinpath('All_Data_with_CGPS_bins.csv'), index_col=0)
centers = pd.read_csv(datadir.joinpath('PC_bin_centers.csv'), index_col=0)

## restrict the treatments and PCs specifically for bootstrapping
bstreats = []#['Random','Galvanotaxis', 'DMSO', 'CK666','Para-Nitro-Blebbistatin']
bspcs = []#[[1,2],[4,5],[2,8]]
alldatabs = True

In [ ]:
############# create all CGPSs #############
for whichpcs in pc_combos:
    #pass if we want to restrict PCs
    if len(bspcs)>0 and whichpcs not in bspcs:
        continue
    if __name__ ==  '__main__':
        ########### get raw transitions and pairs ###########
        rawtrans = DetailedBalance.get_raw_cgps_trajectories(
                FullFrame, #pandas dataframe with all of the cgps binned data
                whichpcs, #which two PCs to use in the cgps [x,y]
                config,
                )

        ########### interpolate all transitions so that only individual transitions are made ###########
        transdf_sep = DetailedBalance.get_interpolated_cgps_trajectories(
                rawtrans, #pandas dataframe with all of the cgps binned data
                whichpcs, #which two PCs to use in the cgps [x,y]
                config,
                )

        ############## get the counts of cells leaving 
        trans_rate_df_sep = DetailedBalance.aggregate_transition_counts(
                transdf_sep, #transdf_sep from get_interpolated_cgps_trajectories
                whichpcs, #which two PCs to use in the cgps [x,y]
                config,
                )

Aggregated transitions
Finished interpolating trajectories
Total time observed in this CGPS was 2287.3143701227186 minutes
Total time observed in this CGPS was 5813.237243932379 minutes
Total time observed in this CGPS was 1395.509183750561 minutes
Total time observed in this CGPS was 2950.0440673274597 minutes
Total time observed in this CGPS was 37.25634146087305 minutes
Total time observed in this CGPS was 1305.6016706618893 minutes
Finished finding transition rates
Aggregated transitions
Finished interpolating trajectories
Total time observed in this CGPS was 2322.73848031811 minutes
Total time observed in this CGPS was 5879.759392289732 minutes
Total time observed in this CGPS was 1394.9636710140503 minutes
Total time observed in this CGPS was 2999.5853521093572 minutes
Total time observed in this CGPS was 35.94185713293227 minutes
Total time observed in this CGPS was 1305.437166600593 minutes
Finished finding transition rates
Aggregated transitions
Finished interpolating trajecto

In [8]:
############# measure aer and cycling frequencies from the observed raw data ###########

for whichpcs in pc_combos:
    #pass if we want to restrict PCs
    if len(bspcs)>0 and whichpcs not in bspcs:
        continue
    #add specific scaling
    xyscaling = [centers[f'PC{whichpcs[0]}'].diff().mean(),centers[f'PC{whichpcs[1]}'].diff().mean()]
    #set the origin to the actual center
    origin = origins[pc_combos.index(whichpcs)]

    ### open the raw transitions
    rawtrans = pd.read_csv(dbdir.joinpath(
            f'PC{whichpcs[0]}-PC{whichpcs[1]}_transitions_separated.csv'), index_col=0)

    #get the area scaling in x and y based on the size of the bins in the cgps
    results = []
    for i, cell in rawtrans.groupby('CellID'):
        #sort data and get continuous transitions in order
        cell, runs = utils.get_consecutive_transitions(cell)
        #measure aer and cycling frequencies for each run of transitions and add to results
        for r in runs:
            c = cell.iloc[r].reset_index(drop=True)
            results.append(DetailedBalance.get_area_enclosing_rate((
                c,
                config.db_params.nbins,
                xyscaling,
                origin,
                )))

    #make a dataframe and save it
    allaers = pd.concat(results, ignore_index = True)
    allaers.to_csv(dbdir.joinpath(
                    f'PC{whichpcs[0]}-PC{whichpcs[1]}_raw_transition_aer_cf.csv'))



good
nope
good
nope
good
nope
good
nope
good
nope
good
nope
good
nope
good
nope
good
nope
good
nope
good
nope
good
nope
good
nope
good
nope
good
nope
good
nope
good
nope
good
nope
good
nope
good
nope
good
nope
good
nope
good
nope
good
nope
good
nope
good
(121177, 10)
good
nope
good
nope


In [ ]:
########### bootstrap transitions and calculate all the aers and cfs around all the pairwise cgps ###############
for whichpcs in pc_combos:
    #pass if we want to restrict PCs
    if len(bspcs)>0 and whichpcs not in bspcs:
        continue
    ## use a separate savedir to bootstrap using all data
    bssavestr = 'alldatabs' if alldatabs else 'separatedatabs'

    if __name__ ==  '__main__':
        #### open the transitions
        rawtrans = pd.read_csv(dbdir.joinpath(
            f'PC{whichpcs[0]}-PC{whichpcs[1]}_transitions_separated.csv'), index_col=0)

        #merge all treatments if bootstrapping with all data
        if alldatabs:
            rawtrans.loc[:,'Treatment'] = 'alldata'

        #restrict to bootstrapped treatments if desired
        if len(bstreats)>0:
            rawtrans = rawtrans[rawtrans.Treatment.isin(bstreats)]


        ############## BOOTSTRAP MANY TRAJECTORIES ##########
        bstrans, bsint, bsframe_sep_full = DetailedBalance.get_bootstrapped_cgps_trajectories(
                rawtrans, #raw transition pairs from get_raw_cgps_trajectories
                whichpcs, #which two PCs to use in the cgps [x,y]
                config,
                bssavestr, #where to save the bootstrapped dataframes
                )


        ############# open average bootstrapped currents ###################
        bsfield_sep = DetailedBalance.get_avg_current_error(
                bsframe_sep_full, #transition rates in the cgps from get_bootstrapped_cgps_trajectories
                whichpcs, #which two PCs to use in the cgps [x,y]
                config,    
                bssavestr, #where to save the aggregated counts
                )


        ############# measure aer and cycling frequencies ###########

        DetailedBalance.get_aer_cf(
            bstrans,
            whichpcs, #which two PCs to use in the cgps [x,y]
            config,
            bssavestr, #where to save calculated aers and cfs
            )



100%|██████████| 3000/3000 [00:37<00:00, 79.06it/s] 


In [2]:
### load config stuff
config = load_config(microscope_type = 'confocal')

# for ali in ['trajectory','trajectory_shape','shape']:

ali = 'shape'

#set alignment
config._alignment = ali
savedir = config.common.savedir
datadir = savedir / 'shape_data'
dbdir = savedir / 'detailed_balance'
npcs = config.common.npcs
ntrans = config.db_params.ntrans
origins = config.db_params.origins
pc_combos = config.common.pc_combos
####### load common directories and data
FullFrame = pd.read_csv(datadir.joinpath('All_Data_with_CGPS_bins.csv'), index_col=0)
centers = pd.read_csv(datadir.joinpath('PC_bin_centers.csv'), index_col=0)

## restrict the treatments and PCs specifically for bootstrapping
bstreats = []#['Random','Galvanotaxis', 'DMSO', 'CK666','Para-Nitro-Blebbistatin']
bspcs = []#[[1,2],[4,5],[2,8]]
alldatabs = True
    

############# create all CGPSs #############
for whichpcs in pc_combos:

    if __name__ ==  '__main__':
        ########### get raw transitions and pairs ###########
        rawtrans = DetailedBalance.get_raw_cgps_trajectories(
                FullFrame, #pandas dataframe with all of the cgps binned data
                whichpcs, #which two PCs to use in the cgps [x,y]
                config,
                )

        ########### interpolate all transitions so that only individual transitions are made ###########
        transdf_sep = DetailedBalance.get_interpolated_cgps_trajectories(
                rawtrans, #pandas dataframe with all of the cgps binned data
                whichpcs, #which two PCs to use in the cgps [x,y]
                config,
                )

        ############## get the counts of cells leaving 
        trans_rate_df_sep = DetailedBalance.aggregate_transition_counts(
                transdf_sep, #transdf_sep from get_interpolated_cgps_trajectories
                whichpcs, #which two PCs to use in the cgps [x,y]
                config,
                )


############# measure aer and cycling frequencies from the observed raw data ###########

for whichpcs in pc_combos:
    
    #add specific scaling
    xyscaling = [centers[f'PC{whichpcs[0]}'].diff().mean(),centers[f'PC{whichpcs[1]}'].diff().mean()]
    #set the origin to the actual center
    origin = origins[pc_combos.index(whichpcs)]

    ### open the raw transitions
    rawtrans = pd.read_csv(dbdir.joinpath(
            f'PC{whichpcs[0]}-PC{whichpcs[1]}_transitions_separated.csv'), index_col=0)

    #get the area scaling in x and y based on the size of the bins in the cgps
    results = []
    for i, cell in rawtrans.groupby('CellID'):
        #sort data and get continuous transitions in order
        cell, runs = utils.get_consecutive_transitions(cell)
        #measure aer and cycling frequencies for each run of transitions and add to results
        for r in runs:
            c = cell.iloc[r].reset_index(drop=True)
            results.append(DetailedBalance.get_area_enclosing_rate((
                c,
                config.db_params.nbins,
                xyscaling,
                origin,
                )))

    #make a dataframe and save it
    allaers = pd.concat(results, ignore_index = True)
    allaers.to_csv(dbdir.joinpath(
                    f'PC{whichpcs[0]}-PC{whichpcs[1]}_raw_transition_aer_cf.csv'))




########### bootstrap transitions and calculate all the aers and cfs around all the pairwise cgps ###############
for whichpcs in pc_combos:
    #pass if we want to restrict PCs
    if len(bspcs)>0 and whichpcs not in bspcs:
        continue
    ## use a separate savedir to bootstrap using all data
    bssavestr = 'alldatabs' if alldatabs else 'separatedatabs'

    if __name__ ==  '__main__':
        #### open the transitions
        rawtrans = pd.read_csv(dbdir.joinpath(
            f'PC{whichpcs[0]}-PC{whichpcs[1]}_transitions_separated.csv'), index_col=0)

        #merge all treatments if bootstrapping with all data
        if alldatabs:
            rawtrans.loc[:,'Treatment'] = 'alldata'

        #restrict to bootstrapped treatments if desired
        if len(bstreats)>0:
            rawtrans = rawtrans[rawtrans.Treatment.isin(bstreats)]


        ############## BOOTSTRAP MANY TRAJECTORIES ##########
        bstrans, bsint, bsframe_sep_full = DetailedBalance.get_bootstrapped_cgps_trajectories(
                rawtrans, #raw transition pairs from get_raw_cgps_trajectories
                whichpcs, #which two PCs to use in the cgps [x,y]
                config,
                bssavestr, #where to save the bootstrapped dataframes
                )


        ############# open average bootstrapped currents ###################
        bsfield_sep = DetailedBalance.get_avg_current_error(
                bsframe_sep_full, #transition rates in the cgps from get_bootstrapped_cgps_trajectories
                whichpcs, #which two PCs to use in the cgps [x,y]
                config,    
                bssavestr, #where to save the aggregated counts
                )


        ############# measure aer and cycling frequencies ###########
        #add specific scaling
        xyscaling = [centers[f'PC{whichpcs[0]}'].diff().mean(),centers[f'PC{whichpcs[1]}'].diff().mean()]
        #set the origin to the actual center
        origin = origins[pc_combos.index(whichpcs)]

        DetailedBalance.get_aer_cf(
            bstrans,
            xyscaling, #scaling of the bins in real units of whatever the CGPS axis parameters are
            origin, #origin in [x bin,y bin]
            whichpcs, #which two PCs to use in the cgps [x,y]
            config,
            bssavestr, #where to save calculated aers and cfs
            )



Aggregated transitions
Finished interpolating trajectories
Total time observed in this CGPS was 2444.7838595268986 minutes
Total time observed in this CGPS was 6060.382312847608 minutes
Total time observed in this CGPS was 1474.3265087913717 minutes
Total time observed in this CGPS was 3034.2402569073874 minutes
Total time observed in this CGPS was 37.665241938674775 minutes
Total time observed in this CGPS was 1347.8786777644307 minutes
Finished finding transition rates
Aggregated transitions
Finished interpolating trajectories
Total time observed in this CGPS was 2584.477638206903 minutes
Total time observed in this CGPS was 6303.566480993661 minutes
Total time observed in this CGPS was 1491.5567826094561 minutes
Total time observed in this CGPS was 3169.755545715004 minutes
Total time observed in this CGPS was 37.42623831925751 minutes
Total time observed in this CGPS was 1364.1894712667452 minutes
Finished finding transition rates
Aggregated transitions
Finished interpolating traje

100%|██████████| 3000/3000 [01:46<00:00, 28.05it/s]


Interpolating trajectories for alldata


100%|██████████| 3000/3000 [00:07<00:00, 385.29it/s] 


Calculating bootstrapped CGPS transition rates for alldata


...

100%|██████████| 3000/3000 [00:09<00:00, 303.04it/s]


Calculating bootstrapped CGPS transition rates for alldata


100%|██████████| 3000/3000 [01:33<00:00, 32.09it/s]


Finished bootstrapping


100%|██████████| 3000/3000 [00:36<00:00, 82.10it/s] 
